# Análisis exploratorio del dataset Contraceptive Method Choice (cmc)
Analizamos el dataset de UCI antes de convertirlo en imágenes con TINTOlib; aquí no limpiamos nada.
La limpieza y el preprocesado ajustado en TRAIN están en `cleanup.py`.

## 0. Configuración
Fijamos semilla, rutas relativas a la carpeta del notebook, esquema de cmc.names y estilo de figuras.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 42
# Ejecutamos el notebook desde su propia carpeta (codigo/dataset_analysis/cmc).
NOTEBOOK_DIR = Path.cwd()
RAW_PATH = NOTEBOOK_DIR / ".." / ".." / "datasets" / "cmc" / "cmc.csv"
assert RAW_PATH.exists(), f"No encontramos {RAW_PATH}: abrimos el notebook desde codigo/dataset_analysis/cmc"
# Huella del cmc.csv que comprobamos valor a valor contra UCI (ucimlrepo id 30).
RAW_SHA256 = "ad2a49da55df24b061a994ecc4d90c856aba35acd05138851aa251c4f06303c9"

COLUMNS = ["wife_age", "wife_edu", "husband_edu", "num_children", "wife_religion", "wife_working",
           "husband_occupation", "standard_of_living_index", "media_exposure", "contraceptive_method"]
TARGET = "contraceptive_method"
FEATURES = COLUMNS[:-1]

# Tipo de cada atributo según cmc.names; las "categóricas" con 1=low..4=high son ordinales.
NUMERIC = ["wife_age", "num_children"]
ORDINAL = ["wife_edu", "husband_edu", "standard_of_living_index"]
BINARY = ["wife_religion", "wife_working", "media_exposure"]
NOMINAL = ["husband_occupation"]
CATEGORICAL = ORDINAL + BINARY + NOMINAL
DOCUMENTED_CODES = {
    "wife_edu": [1, 2, 3, 4],
    "husband_edu": [1, 2, 3, 4],
    "wife_religion": [0, 1],
    "wife_working": [0, 1],
    "husband_occupation": [1, 2, 3, 4],
    "standard_of_living_index": [1, 2, 3, 4],
    "media_exposure": [0, 1],
    TARGET: [1, 2, 3],
}
DOCUMENTED_MEANING = {
    "wife_religion": "0=No islam, 1=Islam",
    "wife_working": "0=Sí trabaja, 1=No trabaja",
    "media_exposure": "0=Buena, 1=No buena",
}
# La encuesta (NICPS 1987) recoge mujeres casadas en edad fértil, de 15 a 49 años.
AGE_RANGE = (15, 49)
CLASSES = [1, 2, 3]
CLASS_LABELS = {1: "Sin método (1)", 2: "Larga duración (2)", 3: "Corta duración (3)"}
CLASS_SHORT = {1: "Ninguno", 2: "Largo", 3: "Corto"}

# Colores fijos por clase en todas las figuras.
CLASS_COLORS = {1: "#159f7c", 2: "#2a78d6", 3: "#eb6834"}
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
GRID = "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": INK_SECONDARY,
    "axes.labelcolor": INK,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "font.size": 9,
    "legend.frameon": False,
})

### Funciones auxiliares
Corrección de Holm, V de Cramér, eta sobre rangos, Kruskal-Wallis con épsilon², post-hoc de Dunn
y residuos ajustados de Haberman, todos pensados para un objetivo de 3 clases.

In [ ]:
def holm_correction(p_values):
    """Corrección de Holm-Bonferroni (controla FWER sin suponer independencia)."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        running_max = max(running_max, (m - rank) * p[idx])
        adjusted[idx] = min(1.0, running_max)
    return adjusted


def cramers_v(x, y):
    """V de Cramér con corrección de sesgo de Bergsma (2013)."""
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    r, k = table.shape
    phi2 = chi2 / n
    phi2_corr = max(0.0, phi2 - (k - 1) * (r - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    k_corr = k - (k - 1) ** 2 / (n - 1)
    denom = min(k_corr - 1, r_corr - 1)
    return np.sqrt(phi2_corr / denom) if denom > 0 else np.nan


def correlation_ratio(categories, values):
    """Razón de correlación eta (sobre rangos) entre una categórica y una numérica u ordinal."""
    # Usamos rangos para que eta sea robusta a las colas y válida con ordinales.
    vals = values.astype(float).rank()
    grand_mean = vals.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for _, g in vals.groupby(categories))
    ss_total = ((vals - grand_mean) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan


def kruskal_epsilon(values, groups):
    """Kruskal-Wallis y su tamaño de efecto épsilon² = H / (n - 1) (Tomczak y Tomczak, 2014)."""
    samples = [values[groups == g] for g in CLASSES]
    h, p = stats.kruskal(*samples)
    return h, p, h / (len(values) - 1)


def dunn_test(values, groups):
    """Post-hoc de Dunn (rangos conjuntos, corrección por empates) con Holm sobre los 3 pares."""
    ranks = values.rank()
    n = len(values)
    tie_counts = values.value_counts().values
    tie_term = (tie_counts ** 3 - tie_counts).sum() / (12 * (n - 1))
    rows = []
    for a, b in [(1, 2), (1, 3), (2, 3)]:
        ra, rb = ranks[groups == a], ranks[groups == b]
        se = np.sqrt((n * (n + 1) / 12 - tie_term) * (1 / len(ra) + 1 / len(rb)))
        z = (ra.mean() - rb.mean()) / se
        rows.append({"par": f"{a}-{b}", "z": z, "p": 2 * stats.norm.sf(abs(z))})
    result = pd.DataFrame(rows).set_index("par")
    result["p_Holm"] = holm_correction(result["p"])
    return result


def adjusted_residuals(table):
    """Residuos estandarizados ajustados de Haberman: ~N(0, 1) en cada celda bajo independencia."""
    observed = table.values.astype(float)
    n = observed.sum()
    rows = observed.sum(axis=1, keepdims=True)
    cols = observed.sum(axis=0, keepdims=True)
    expected = rows @ cols / n
    residuals = (observed - expected) / np.sqrt(expected * (1 - rows / n) * (1 - cols / n))
    return pd.DataFrame(residuals, index=table.index, columns=table.columns)

## 1. Carga y procedencia
Leemos sin cabecera porque cmc.csv es una copia de cmc.data de UCI, que no la tiene.
Comprobamos su huella SHA-256 y exigimos que todo sea entero, porque cmc.names solo define códigos enteros.

In [ ]:
digest = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print(f"Fichero: {RAW_PATH.resolve().name} | SHA-256 {digest}")
print(f"Coincide con la copia verificada contra UCI (ucimlrepo id 30): {digest == RAW_SHA256}")
raw = pd.read_csv(RAW_PATH, header=None, names=COLUMNS, dtype=str, keep_default_na=False)
print(f"Forma cruda: {raw.shape} (documentado en cmc.names: 1473 x 10)")
assert raw.shape == (1473, 10), "El fichero no coincide con la documentación de UCI"
non_integer = int((~raw.apply(lambda s: s.str.fullmatch(r"-?\d+"))).sum().sum())
print(f"Celdas no enteras (vacías, '?', texto o decimales): {non_integer}")
df = raw.astype(int)
print("\nPrimeras filas:")
print(df.head().to_string())

## 2. Esquema y tipos
Clasificamos cada atributo según cmc.names, porque el tipo decide el test, la codificación y la figura.

In [ ]:
kinds = {**{c: "numérica" for c in NUMERIC}, **{c: "ordinal (1=bajo..4=alto)" for c in ORDINAL},
         **{c: "binaria" for c in BINARY}, **{c: "nominal (sin orden documentado)" for c in NOMINAL},
         TARGET: "objetivo nominal (3 clases)"}
rows = [{"columna": c, "tipo": kinds[c], "dtype": str(df[c].dtype), "n_unicos": df[c].nunique(),
         "min": df[c].min(), "max": df[c].max()} for c in COLUMNS]
print(pd.DataFrame(rows).set_index("columna").to_string())
print("\nSignificado de las binarias (cmc.names):", DOCUMENTED_MEANING)
print("Ojo: en wife_working y media_exposure el 1 es la respuesta negativa ('No', 'No buena').")

## 3. Valores ausentes
cmc.names declara que no hay ausentes; aun así buscamos NaN y centinelas (0 en códigos 1..4, 99, negativos).

In [ ]:
print(f"NaN totales: {int(df.isna().sum().sum())}")
sentinels = {c: int(((df[c] == 0) & (min(DOCUMENTED_CODES.get(c, [0])) == 1)).sum()
                    + (df[c] >= 99).sum() + (df[c] < 0).sum()) for c in COLUMNS}
print(f"Posibles centinelas por columna: {sentinels}")
print("Conclusión: no hay ausentes, así que la imputación por partición solo actúa como red de seguridad.")

## 4. Códigos inválidos y consistencia
Comprobamos cada código contra cmc.names, porque uno fuera del esquema sería un error de transcripción.

In [ ]:
for col, codes in DOCUMENTED_CODES.items():
    invalid = int((~df[col].isin(codes)).sum())
    unused = sorted(set(codes) - set(df[col].unique()))
    print(f"  {col}: fuera del esquema={invalid}; códigos documentados sin uso={unused or 'ninguno'}")
age_low, age_high = AGE_RANGE
print(f"\nwife_age fuera de [{age_low}, {age_high}]: {int((~df['wife_age'].between(age_low, age_high)).sum())}"
      f" (observado {df['wife_age'].min()}-{df['wife_age'].max()})")
print(f"num_children negativos: {int((df['num_children'] < 0).sum())}; máximo: {df['num_children'].max()}")

Contrastamos hijos frente a edad: más hijos que años desde los 13 sería biológicamente dudoso.
También miramos el objetivo de las mujeres sin hijos, porque casi ninguna debería usar un método de larga duración.

In [ ]:
years_fertile = (df["wife_age"] - 13).clip(lower=1)
implausible = df[df["num_children"] > years_fertile]
print(f"Filas con num_children > edad - 13: {len(implausible)}")
if len(implausible):
    print(implausible[["wife_age", "num_children"]].to_string())
print("Máximo de hijos por edad (16-25):", df.groupby("wife_age")["num_children"].max().loc[16:25].to_dict())
print("\nnum_children == 0 frente al objetivo:", df.loc[df["num_children"] == 0, TARGET].value_counts()
      .reindex(CLASSES, fill_value=0).to_dict())

## 5. Duplicados, columnas casi constantes y cardinalidad
Separamos duplicados exactos y de atributos: estos últimos revelan perfiles con etiquetas distintas.
Si fueran copias de transcripción aparecerían contiguas, así que medimos su distancia en el fichero.

In [ ]:
exact = df.duplicated()
in_groups = df.duplicated(keep=False)
print(f"Filas duplicadas exactas: {int(exact.sum())} ({100 * exact.mean():.1f} %), "
      f"en {df[in_groups].groupby(COLUMNS).ngroups} grupos que suman {int(in_groups.sum())} filas")
feature_dups = df.duplicated(subset=FEATURES)
profiles = df.groupby(FEATURES)[TARGET].nunique()
print(f"Filas duplicadas en atributos (ignorando el objetivo): {int(feature_dups.sum())}")
print(f"Perfiles distintos: {len(profiles)}; perfiles con etiquetas contradictorias: {int((profiles > 1).sum())}")
gaps = np.concatenate([np.diff(g.index.values) for _, g in df[in_groups].groupby(COLUMNS)])
print(f"Distancia entre copias en el fichero: mínima {gaps.min()}, mediana {np.median(gaps):.0f}, "
      f"contiguas {int((gaps == 1).sum())}")
print("Perfil medio de las filas duplicadas frente al global:")
print(pd.DataFrame({"duplicadas": df[exact].mean(), "global": df.mean()}).round(2).T.to_string())
print(f"Clases de las copias: {df.loc[exact, TARGET].value_counts().reindex(CLASSES).to_dict()}")

Contrastamos copias frente a mujeres distintas: una copia repite siempre la etiqueta, y dos mujeres
con el mismo perfil coinciden con probabilidad sum_k p_k(x)², que estimamos fuera de pliegue (GroupKFold por perfil).
Un bootstrap por perfiles, con semilla fija, da el intervalo de la fracción de copias.

In [ ]:
profile_groups = df.groupby(FEATURES).ngroup()
model = make_pipeline(OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=2000))
proba = cross_val_predict(model, df[FEATURES], df[TARGET], groups=profile_groups, cv=GroupKFold(n_splits=5),
                          method="predict_proba")
agreement = pd.Series((proba ** 2).sum(axis=1), index=df.index)
per_group = []
for _, g in df[df.duplicated(subset=FEATURES, keep=False)].groupby(FEATURES):
    labels = g[TARGET].values
    n_pairs = len(g) * (len(g) - 1) / 2
    same = sum(labels[i] == labels[j] for i in range(len(g)) for j in range(i + 1, len(g)))
    per_group.append((n_pairs, same, n_pairs * agreement[g.index].mean()))
per_group = np.array(per_group, dtype=float)


def copy_fraction(rows):
    obs, exp = rows[:, 1].sum() / rows[:, 0].sum(), rows[:, 2].sum() / rows[:, 0].sum()
    return obs, exp, (obs - exp) / (1 - exp)


obs, exp, frac = copy_fraction(per_group)
rng = np.random.default_rng(SEED)
boot = [copy_fraction(per_group[rng.integers(0, len(per_group), len(per_group))])[2] for _ in range(2000)]
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"Pares con el mismo perfil: {int(per_group[:, 0].sum())} en {len(per_group)} perfiles; misma etiqueta "
      f"observada {obs:.3f} frente a {exp:.3f} esperada si son mujeres distintas")
print(f"Fracción de copias implícita: {frac:.3f} (IC bootstrap 95 % [{ci_low:.3f}, {ci_high:.3f}])")

Medimos la frecuencia del valor dominante: por encima del 95 % una columna aporta poca variación.

In [ ]:
rows = []
for col in COLUMNS:
    counts = df[col].value_counts(normalize=True)
    rows.append({"columna": col, "n_unicos": df[col].nunique(), "valor_dominante": counts.index[0],
                 "%_dominante": round(100 * counts.iloc[0], 1)})
cardinality = pd.DataFrame(rows).set_index("columna")
print(cardinality.to_string())
near_constant = cardinality[cardinality["%_dominante"] > 95].index.tolist()
print(f"Columnas casi constantes (>95 %): {near_constant or 'ninguna'}")

## 6. Estadística univariante: numéricas
Describimos wife_age y num_children y medimos cuánto reducirían la asimetría sqrt y log1p.

In [ ]:
rows = []
for col in NUMERIC:
    x = df[col]
    q1, q3 = x.quantile([0.25, 0.75])
    rows.append({"col": col, "media": x.mean(), "mediana": x.median(), "std": x.std(), "min": x.min(),
                 "Q1": q1, "Q3": q3, "max": x.max(), "asimetría": stats.skew(x),
                 "curtosis_exc": stats.kurtosis(x), "asim_sqrt": stats.skew(np.sqrt(x)),
                 "asim_log1p": stats.skew(np.log1p(x))})
with pd.option_context("display.float_format", "{:.3f}".format):
    print(pd.DataFrame(rows).set_index("col").to_string())

Contrastamos normalidad con Shapiro-Wilk y D'Agostino K² para elegir entre tests paramétricos o de rangos.
Para atípicos usamos la regla IQR (robusta) y |z| > 3 solo como referencia, porque num_children es asimétrica.

In [ ]:
print("\nNormalidad:")
for col in NUMERIC:
    x = df[col]
    print(f"  {col}: Shapiro W={stats.shapiro(x).statistic:.3f} p={stats.shapiro(x).pvalue:.2e} | "
          f"D'Agostino K²={stats.normaltest(x).statistic:.1f} p={stats.normaltest(x).pvalue:.2e}")
print("\nAtípicos por regla IQR (1.5·IQR) y |z| > 3:")
for col in NUMERIC:
    x = df[col]
    q1, q3 = x.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    iqr_out = x[(x < q1 - 1.5 * (q3 - q1)) | (x > upper)]
    print(f"  {col}: IQR={len(iqr_out)} ({100 * len(iqr_out) / len(x):.1f} %, umbral superior {upper:.1f}), "
          f"|z|>3={int((np.abs(stats.zscore(x)) > 3).sum())}, valores atípicos={sorted(int(v) for v in iqr_out.unique())}")
print("Cola de num_children:", df["num_children"].value_counts().sort_index().loc[9:].to_dict())

Dibujamos los histogramas con un bin por valor entero y marcamos la mediana.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, col, label in [(axes[0], "wife_age", "Edad de la mujer (años)"),
                       (axes[1], "num_children", "Nº de hijos nacidos")]:
    x = df[col]
    bins = np.arange(x.min() - 0.5, x.max() + 1.5, 1)
    ax.hist(x, bins=bins, color=CLASS_COLORS[2], alpha=0.6, edgecolor="white", linewidth=0.4)
    ax.axvline(x.median(), color=CLASS_COLORS[3], linestyle="--", linewidth=1.5, label=f"Mediana = {x.median():.0f}")
    ax.set_title(f"{col} (asimetría {stats.skew(x):.2f})")
    ax.set_xlabel(label)
    ax.set_ylabel("Nº de mujeres")
    ax.legend()
fig.suptitle("Distribución de las variables numéricas", fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Estadística univariante: ordinales, binarias y nominal
Contamos cada nivel y señalamos los raros (<5 %), porque generan columnas one-hot casi vacías.

In [ ]:
for col in CATEGORICAL:
    counts = df[col].value_counts().sort_index()
    pct = (100 * counts / len(df)).round(1)
    print(f"  {col}: " + ", ".join(f"{k}={v} ({p} %)" for k, v, p in zip(counts.index, counts, pct)))
rare = {c: df[c].value_counts()[lambda s: s < 0.05 * len(df)].to_dict() for c in CATEGORICAL}
print(f"Niveles con <5 % de filas: { {c: v for c, v in rare.items() if v} }")

Usamos barras apiladas al 100 % por nivel, porque con 3 clases es lo que mejor muestra la tendencia.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.8))
for ax, col in zip(axes.flat, CATEGORICAL):
    table = pd.crosstab(df[col], df[TARGET]).reindex(columns=CLASSES, fill_value=0)
    shares = table.div(table.sum(axis=1), axis=0)
    bottom = np.zeros(len(shares))
    positions = np.arange(len(shares))
    for cls in CLASSES:
        ax.bar(positions, shares[cls], bottom=bottom, color=CLASS_COLORS[cls], width=0.7,
               label=CLASS_LABELS[cls], edgecolor="white", linewidth=0.6)
        bottom += shares[cls].values
    ax.set_xticks(positions, [f"{lvl}\nn={n}" for lvl, n in zip(table.index, table.sum(axis=1))], fontsize=7.5)
    ax.set_ylim(0, 1)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("Proporción")
    ax.grid(axis="x", visible=False)
ax = axes.flat[-1]
ax.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COLORS[c]) for c in CLASSES]
ax.legend(handles, [CLASS_LABELS[c] for c in CLASSES], loc="center", title="Método anticonceptivo")
fig.suptitle("Método anticonceptivo por nivel de cada atributo categórico (barras al 100 %)", fontweight="bold")
fig.tight_layout()
plt.show()

## 8. Balance del objetivo
Con un ratio < 2 no remuestreamos, pero estratificamos y medimos con F1 macro además de accuracy.

In [ ]:
class_counts = df[TARGET].value_counts().reindex(CLASSES)
print(f"Clases: {class_counts.to_dict()} | proporciones: {(100 * class_counts / len(df)).round(1).to_dict()}")
print(f"Ratio mayoritaria/minoritaria = {class_counts.max() / class_counts.min():.2f}; "
      f"tasa de acierto de la clase mayoritaria = {class_counts.max() / len(df):.3f}")

fig, ax = plt.subplots(figsize=(5.6, 3.6))
bars = ax.bar([CLASS_LABELS[c] for c in CLASSES], class_counts.values,
              color=[CLASS_COLORS[c] for c in CLASSES], width=0.55)
for bar, value in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 8, f"{value} ({100 * value / len(df):.1f} %)",
            ha="center", va="bottom", color=INK)
ax.set_ylabel("Nº de mujeres")
ax.set_title("Balance de la variable objetivo contraceptive_method")
ax.set_ylim(0, class_counts.max() * 1.15)
plt.show()

## 9. Bivariante: numéricas frente al objetivo
Usamos Kruskal-Wallis (no ANOVA) porque no hay normalidad y el objetivo tiene 3 grupos independientes.
Dunn con Holm localiza después qué pares de clases difieren, que es lo que Kruskal-Wallis no dice.

In [ ]:
y = df[TARGET]
rows = []
for col in NUMERIC:
    h, p, eps2 = kruskal_epsilon(df[col], y)
    medians = df.groupby(TARGET)[col].median().to_dict()
    rows.append({"col": col, **{f"mediana_{c}": medians[c] for c in CLASSES}, "H": h, "p": p, "épsilon²": eps2})
numeric_vs_target = pd.DataFrame(rows).set_index("col")
numeric_vs_target["p_Holm"] = holm_correction(numeric_vs_target["p"])
with pd.option_context("display.float_format", "{:.4g}".format):
    print(numeric_vs_target.to_string())
print("(épsilon² ~0.01 pequeño, ~0.08 medio, ~0.26 grande)")
for col in NUMERIC:
    print(f"\nDunn ({col}):")
    with pd.option_context("display.float_format", "{:.3g}".format):
        print(dunn_test(df[col], y).to_string())

Combinamos violín y caja por clase para ver forma y mediana a la vez.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, NUMERIC):
    data = [df.loc[y == c, col] for c in CLASSES]
    parts = ax.violinplot(data, showextrema=False, widths=0.8)
    for body, cls in zip(parts["bodies"], CLASSES):
        body.set_facecolor(CLASS_COLORS[cls])
        body.set_alpha(0.35)
    box = ax.boxplot(data, widths=0.25, patch_artist=True, medianprops={"color": INK, "linewidth": 1.5},
                     flierprops={"marker": "o", "markersize": 2.5, "alpha": 0.5, "markeredgecolor": INK_SECONDARY})
    for patch, cls in zip(box["boxes"], CLASSES):
        patch.set_facecolor(CLASS_COLORS[cls])
        patch.set_alpha(0.85)
    ax.set_xticks([1, 2, 3], [CLASS_LABELS[c] for c in CLASSES])
    ax.set_ylabel(col)
    row = numeric_vs_target.loc[col]
    ax.set_title(f"{col}: ε² = {row['épsilon²']:.3f}, p Holm = {row['p_Holm']:.1e}")
fig.suptitle("Variables numéricas por método anticonceptivo (Kruskal-Wallis)", fontweight="bold")
fig.tight_layout()
plt.show()

Dibujamos la proporción de cada método por tramo de edad y por nº de hijos, porque la relación con la edad no es monótona.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
age_bins = pd.cut(df["wife_age"], bins=[15, 20, 25, 30, 35, 40, 45, 49], include_lowest=True,
                  labels=["15-20", "21-25", "26-30", "31-35", "36-40", "41-45", "46-49"])
child_bins = df["num_children"].clip(upper=8)
for ax, groups, xlabel in [(axes[0], age_bins, "Edad de la mujer (años)"),
                           (axes[1], child_bins, "Nº de hijos (8 = 8 o más)")]:
    table = pd.crosstab(groups, y).reindex(columns=CLASSES, fill_value=0)
    shares = table.div(table.sum(axis=1), axis=0)
    labels = [str(i) for i in table.index]
    for cls in CLASSES:
        ax.plot(range(len(shares)), shares[cls], marker="o", color=CLASS_COLORS[cls], label=CLASS_LABELS[cls])
    ax.set_xticks(range(len(shares)), [f"{l}\nn={n}" for l, n in zip(labels, table.sum(axis=1))], fontsize=7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Proporción dentro del grupo")
    ax.set_ylim(0, 1)
axes[0].legend(loc="upper left")
fig.suptitle("Método anticonceptivo según edad y número de hijos", fontweight="bold")
fig.tight_layout()
plt.show()
print("\nMétodo por tramo de edad (%):")
print((100 * pd.crosstab(age_bins, y, normalize="index")).round(1).to_string())

## 10. Bivariante: categóricas frente al objetivo
Usamos chi-cuadrado de independencia (k x 3) con V de Cramér corregida.
Para las ordinales añadimos Kruskal-Wallis, que sí usa el orden de los códigos.

In [ ]:
rows = []
for col in CATEGORICAL:
    table = pd.crosstab(df[col], y)
    chi2, p, dof, expected = stats.chi2_contingency(table, correction=False)
    row = {"col": col, "niveles": table.shape[0], "chi2": chi2, "gl": dof,
           "min_esperada": expected.min(), "p_chi2": p, "V_Cramer": cramers_v(df[col], y)}
    if col in ORDINAL:
        _, p_kw, eps2 = kruskal_epsilon(df[col], y)
        row.update({"p_KW": p_kw, "épsilon²": eps2})
    rows.append(row)
categorical_vs_target = pd.DataFrame(rows).set_index("col")
categorical_vs_target["p_chi2_Holm"] = holm_correction(categorical_vs_target["p_chi2"])
with pd.option_context("display.float_format", "{:.4g}".format):
    print(categorical_vs_target.to_string())
print("(todas las frecuencias esperadas >= 5, así que la aproximación chi-cuadrado es válida;"
      "\n V de Cramér: con 4 niveles (gl* = 2) ~0.07 pequeño, ~0.21 medio, ~0.35 grande;"
      "\n con las binarias (gl* = 1) ~0.10 pequeño, ~0.30 medio, ~0.50 grande)")

Como post-hoc calculamos los residuos ajustados con Holm dentro de cada tabla, para saber qué celdas causan la asociación.

In [ ]:
residual_tables = {}
print("\nCeldas significativas (|residuo ajustado|, Holm < 0.05):")
for col in CATEGORICAL:
    residuals = adjusted_residuals(pd.crosstab(df[col], y))
    p = 2 * stats.norm.sf(np.abs(residuals.values)).ravel()
    p_adj = holm_correction(p).reshape(residuals.shape)
    residual_tables[col] = (residuals, p_adj)
    hits = [f"{col}={lvl}/{CLASS_SHORT[cls]}: {residuals.loc[lvl, cls]:+.1f}"
            for i, lvl in enumerate(residuals.index) for j, cls in enumerate(CLASSES) if p_adj[i, j] < 0.05]
    print(f"  {col}: " + ("; ".join(hits) if hits else "ninguna"))

Mostramos los residuos como mapas de calor, con asterisco en las celdas significativas.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 7), gridspec_kw={"hspace": 0.55})
for ax, col in zip(axes.flat, CATEGORICAL):
    residuals, p_adj = residual_tables[col]
    im = ax.imshow(residuals.values, cmap="RdBu_r", vmin=-8, vmax=8, aspect="auto")
    for i in range(residuals.shape[0]):
        for j in range(residuals.shape[1]):
            mark = "*" if p_adj[i, j] < 0.05 else ""
            ax.text(j, i, f"{residuals.values[i, j]:+.1f}{mark}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(residuals.values[i, j]) > 5 else INK)
    ax.set_xticks(range(3), [CLASS_SHORT[c] for c in CLASSES])
    ax.set_yticks(range(len(residuals)), residuals.index)
    ax.set_title(f"{col}\nV = {categorical_vs_target.loc[col, 'V_Cramer']:.2f}", fontsize=8.5)
    ax.grid(False)
axes.flat[-1].axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="Residuo estandarizado ajustado")
fig.suptitle("Residuos ajustados nivel x método (* = Holm < 0.05; rojo = más casos de lo esperado)",
             fontweight="bold")
plt.show()

## 11. Correlaciones entre numéricas y ordinales, y colinealidad
Preferimos Spearman porque las ordinales solo tienen orden y las numéricas no son normales.
El VIF sobre todos los atributos (nominal en one-hot) detecta colinealidad multivariante.

In [ ]:
ordered = NUMERIC + ORDINAL
spearman = df[ordered].corr(method="spearman")
kendall = df[ordered].corr(method="kendall")
with pd.option_context("display.float_format", "{:.3f}".format):
    print("Spearman:\n" + spearman.to_string())
    print("Kendall tau-b (referencia con empates):\n" + kendall.to_string())
upper = spearman.where(np.triu(np.ones(spearman.shape, dtype=bool), k=1)).stack()
print(f"Máximo |Spearman| entre pares: {upper.abs().max():.3f} ({upper.abs().idxmax()})")
X = pd.get_dummies(df[FEATURES], columns=NOMINAL, drop_first=True, dtype=float)
vif = {}
for col in X.columns:
    target = X[col].values
    design = np.column_stack([np.ones(len(X)), X.drop(columns=col).values])
    beta, *_ = np.linalg.lstsq(design, target, rcond=None)
    r2 = 1 - ((target - design @ beta) ** 2).sum() / ((target - target.mean()) ** 2).sum()
    vif[col] = 1 / (1 - r2)
print("VIF:", {k: round(float(v), 2) for k, v in vif.items()}, "(> 5 sería preocupante)")

Dibujamos la matriz de Spearman con su valor en cada celda.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(spearman.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(ordered)), ordered, rotation=35, ha="right")
ax.set_yticks(range(len(ordered)), ordered)
for i in range(len(ordered)):
    for j in range(len(ordered)):
        value = spearman.values[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(value) > 0.6 else INK)
ax.set_title("Correlación de Spearman (numéricas y ordinales)")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="rho de Spearman")
plt.show()

## 12. Asociaciones entre todas las variables
Usamos medidas en [0, 1]: |Spearman| entre ordenadas, V de Cramér entre no ordenadas y eta sobre rangos en el resto.

In [ ]:
ordered_set = set(NUMERIC + ORDINAL)
unordered_set = set(BINARY + NOMINAL + [TARGET])
association = pd.DataFrame(np.eye(len(COLUMNS)), index=COLUMNS, columns=COLUMNS)
for i, a in enumerate(COLUMNS):
    for b in COLUMNS[i + 1:]:
        if a in ordered_set and b in ordered_set:
            value = abs(df[[a, b]].corr(method="spearman").iloc[0, 1])
        elif a in unordered_set and b in unordered_set:
            value = cramers_v(df[a], df[b])
        else:
            cat, num = (a, b) if a in unordered_set else (b, a)
            value = correlation_ratio(df[cat], df[num])
        association.loc[a, b] = association.loc[b, a] = value
pairs = association.where(np.triu(np.ones(association.shape, dtype=bool), k=1)).stack().sort_values(ascending=False)
print("Pares más asociados:")
print(pairs.head(10).round(3).to_string())

Mostramos la matriz completa, objetivo incluido, para ver de un vistazo redundancias y señal.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(association.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(COLUMNS)), COLUMNS, rotation=40, ha="right")
ax.set_yticks(range(len(COLUMNS)), COLUMNS)
for i in range(len(COLUMNS)):
    for j in range(len(COLUMNS)):
        if i != j:
            value = association.values[i, j]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if value > 0.55 else INK)
ax.set_title("Asociación entre variables\n(|Spearman| ordenadas, V de Cramér no ordenadas, eta sobre rangos mixtas)")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="Fuerza de asociación [0, 1]")
plt.show()

## 13. Interacción edad x hijos por clase
Ambas variables son discretas, así que añadimos un pequeño ruido solo al dibujar para que los puntos no se solapen.
Después cuantificamos la pauta con los hijos por año de edad fértil, mediana por clase.

In [ ]:
jitter_rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, cls in zip(axes, CLASSES):
    sub = df[df[TARGET] == cls]
    ax.scatter(sub["wife_age"] + jitter_rng.uniform(-0.35, 0.35, len(sub)),
               sub["num_children"] + jitter_rng.uniform(-0.3, 0.3, len(sub)),
               s=5, alpha=0.45, color=CLASS_COLORS[cls], linewidths=0)
    ax.set_title(f"{CLASS_LABELS[cls]} (n = {len(sub)})")
    ax.set_xlabel("Edad de la mujer (años)")
axes[0].set_ylabel("Nº de hijos")
fig.suptitle("Edad frente a número de hijos por método (con ruido de dibujo)", fontweight="bold")
fig.tight_layout()
plt.show()
rate = df["num_children"] / (df["wife_age"] - 13).clip(lower=1)
print("Mediana de hijos por año desde los 13, por clase:", rate.groupby(df[TARGET]).median().round(3).to_dict())

## 14. Información mutua con el objetivo
La información mutua capta dependencias no lineales y da una referencia de importancia para comparar con XAI.
Tratamos como discretas todas las columnas salvo wife_age y num_children.

In [ ]:
discrete = [c not in NUMERIC for c in FEATURES]
mi = pd.Series(mutual_info_classif(df[FEATURES], df[TARGET], discrete_features=discrete, random_state=SEED),
               index=FEATURES).sort_values(ascending=False)
print(mi.round(4).to_string())
print(f"Entropía del objetivo: {stats.entropy(df[TARGET].value_counts(normalize=True)):.4f} nats")

Ordenamos la información mutua en un gráfico de barras para usarla como referencia frente a XAI.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.barh(mi.index[::-1], mi.values[::-1], color=CLASS_COLORS[2], height=0.6)
for k, v in enumerate(mi.values[::-1]):
    ax.text(v + 0.001, k, f"{v:.3f}", va="center", fontsize=7.5, color=INK)
ax.set_xlabel("Información mutua con contraceptive_method (nats)")
ax.set_title("Información mutua de cada atributo con el objetivo")
ax.grid(axis="y", visible=False)
plt.show()

## Conclusiones
Resumimos lo que el análisis implica para la limpieza (`cleanup.py`) y para la conversión con TINTOlib.

### Procedencia y calidad
- El fichero coincide con cmc.data de UCI (1473 x 10, huella SHA-256 fijada); todo son enteros, sin ausentes ni códigos fuera de cmc.names.
- Las edades van de 16 a 49 y ninguna mujer tiene más hijos que años desde los 13; 95 de las 97 sin hijos no usan método.
- No hay columnas casi constantes; los niveles raros son husband_occupation 4 (27 filas) y husband_edu 1 (44 filas).

### Duplicados
- Hay 48 duplicados exactos (3.3 %), ninguno contiguo, y 62 perfiles con etiquetas contradictorias.
- Los perfiles repetidos coinciden en la etiqueta 0.382 frente a 0.416 esperado (copias -0.06, IC 95 % [-0.21, 0.09]): sin indicios de copias, aunque no descartamos hasta un 9 %.
- Conservamos las 1473 filas como mujeres distintas; en la partición de demostración de `cleanup.py` (80/20, semilla 42) el 5.4 % del test tiene copia exacta en train.

### Objetivo y distribuciones
- Las clases son 629 / 333 / 511 (ratio 1.89): estratificamos las particiones y medimos F1 macro (base mayoritaria 0.427).
- Ninguna numérica es normal; num_children tiene asimetría 1.10 y 45 atípicos IQR (9 a 16 hijos), plausibles, así que no la transformamos.
- No hay redundancia: el mayor par es wife_edu con husband_edu (rho = 0.63) y todos los VIF son <= 1.93.

### Señal frente al objetivo
- La señal es modesta: wife_edu (V = 0.21) y num_children (IM 0.052) lideran; wife_working no se asocia (Holm p = 0.075).
- La edad no es monótona: la corta duración alcanza su máximo entre 21 y 35 años, la larga entre 41 y 45, y en los extremos predomina no usar método; una lectura lineal la infravalora.
- Estas importancias sirven de referencia para contrastar las explicaciones XAI de la CNN.